# Improved Barnes Maze Analysis — Figures for PI Presentation

Three key improvements:
1. **Probe accuracy** replaces goal-box accuracy (floor effect fix)
2. **Learning curve** across all 6 trials (6x more data)
3. **Effect sizes and power** (are our nulls informative?)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='talk', font_scale=1.0)
PAL = {'CTR': '#4C72B0', 'ISF': '#DD8452'}

def clean_colnames(df):
    df = df.copy()
    df.columns = (df.columns.astype(str).str.strip()
        .str.replace(r'[^\w]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True).str.strip('_'))
    return df

# Load
circ = clean_colnames(pd.read_csv('Circadian_raw.csv'))
barnes = clean_colnames(pd.read_csv('Barnes_clean.csv'))
nor = clean_colnames(pd.read_csv('UCBAge_Novel_clean.csv'))
effects = pd.read_csv('effect_sizes_and_power.csv')

if 'Animal_ID' in nor.columns and 'ID' not in nor.columns:
    nor = nor.rename(columns={'Animal_ID': 'ID'})

for df in (circ, barnes, nor):
    df['ID'] = pd.to_numeric(df['ID'], errors='coerce').astype('Int64')

barnes['Trial'] = pd.to_numeric(barnes['Trial'], errors='coerce')
barnes['EntryZone_freq_new'] = pd.to_numeric(barnes['EntryZone_freq_new'], errors='coerce')
barnes['Hole_errors'] = pd.to_numeric(barnes['Hole_errors'], errors='coerce')
barnes['Goal_Box_feq_new'] = pd.to_numeric(barnes['Goal_Box_feq_new'], errors='coerce')
barnes['total_pokes'] = barnes['EntryZone_freq_new'] + barnes['Hole_errors']
barnes['probe_accuracy'] = np.where(barnes['total_pokes'] > 0,
    barnes['EntryZone_freq_new'] / barnes['total_pokes'], np.nan)
barnes['goal_accuracy'] = np.where(
    (barnes['Goal_Box_feq_new'] + barnes['Hole_errors']) > 0,
    barnes['Goal_Box_feq_new'] / (barnes['Goal_Box_feq_new'] + barnes['Hole_errors']),
    np.nan)

print(f'Barnes: {barnes["ID"].nunique()} mice, {len(barnes)} trial-rows')

## Figure 1: Why We Changed the Primary Endpoint
Goal-box accuracy (old) vs probe accuracy (new) at Trial 6

In [ ]:
bt6 = barnes[barnes['Trial'] == 6].copy()
bt6['Light_new'] = bt6['Light_new'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Old endpoint (goal box accuracy) — floor effect
ax = axes[0]
sns.stripplot(data=bt6, x='Light_new', y='goal_accuracy', palette=PAL,
              jitter=0.2, alpha=0.6, size=6, ax=ax)
sns.boxplot(data=bt6, x='Light_new', y='goal_accuracy', palette=PAL,
            width=0.4, fliersize=0, boxprops=dict(alpha=0.3), ax=ax)
ax.set_title('A. Old endpoint: Goal Box accuracy\n(floor effect — max = 1 entry)', fontsize=13)
ax.set_ylabel('Goal Box accuracy\nGoal_Box / (Goal_Box + Errors)')
ax.set_xlabel('Light Group')
ax.set_ylim(-0.02, 0.15)
ax.text(0.5, 0.92, f'97.8% of mice = 0', transform=ax.transAxes,
        ha='center', fontsize=11, color='red', fontweight='bold')

# Panel B: New endpoint (probe accuracy) — real variance
ax = axes[1]
sns.stripplot(data=bt6, x='Light_new', y='probe_accuracy', palette=PAL,
              jitter=0.2, alpha=0.6, size=6, ax=ax)
sns.boxplot(data=bt6, x='Light_new', y='probe_accuracy', palette=PAL,
            width=0.4, fliersize=0, boxprops=dict(alpha=0.3), ax=ax)
ax.set_title('B. New endpoint: Probe accuracy\n(target zone entries — real variance)', fontsize=13)
ax.set_ylabel('Probe accuracy\nEntryZone / (EntryZone + Errors)')
ax.set_xlabel('Light Group')
ax.set_ylim(-0.02, 0.75)

# Add stats
ctr = bt6[bt6['Light_new']=='CTR']['probe_accuracy'].dropna()
isf = bt6[bt6['Light_new']=='ISF']['probe_accuracy'].dropna()
u, p = stats.mannwhitneyu(ctr, isf)
ax.text(0.5, 0.92, f'Mann-Whitney p = {p:.3f} (ns)', transform=ax.transAxes,
        ha='center', fontsize=11)

fig.suptitle('Trial 6: Why Probe Accuracy is the Better Endpoint', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_endpoint_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2: Learning Curve Across All 6 Trials
Mean probe accuracy (+/- SE) by trial and light group, with individual mouse trajectories

In [ ]:
lc = barnes.dropna(subset=['probe_accuracy']).copy()
lc['Light_new'] = lc['Light_new'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Group means with SE
ax = axes[0]
sns.pointplot(data=lc, x='Trial', y='probe_accuracy', hue='Light_new',
              palette=PAL, errorbar='se', dodge=0.1, markers=['o','s'],
              capsize=0.05, ax=ax)
ax.set_title('A. Learning curve: Mean +/- SE', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Trial')
ax.set_ylim(0, 0.55)
ax.legend(title='Light Group')

# Add LME result annotation
ax.text(0.02, 0.95, 'Trial x Light interaction:\nbeta = 0.015, p = 0.240 (ns)',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Panel B: Individual mouse trajectories + group mean
ax = axes[1]
for light, color in PAL.items():
    sub = lc[lc['Light_new'] == light]
    for mid in sub['ID'].unique():
        dm = sub[sub['ID'] == mid].sort_values('Trial')
        ax.plot(dm['Trial'], dm['probe_accuracy'], color=color, alpha=0.08, lw=0.8)

# Overlay group means
means = lc.groupby(['Trial', 'Light_new'])['probe_accuracy'].mean().reset_index()
for light, color in PAL.items():
    sub = means[means['Light_new'] == light]
    ax.plot(sub['Trial'], sub['probe_accuracy'], color=color, lw=3, marker='o',
            markersize=8, label=light, zorder=5)

ax.set_title('B. Individual trajectories + group mean', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Trial')
ax.set_ylim(0, 0.85)
ax.legend(title='Light Group')

fig.suptitle('Spatial Learning Across Trials: No ISF Effect on Learning Rate',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3: Probe Accuracy by Age Group
Age is the strongest predictor of Barnes performance

In [ ]:
bt6 = barnes[barnes['Trial'] == 6].dropna(subset=['probe_accuracy']).copy()
bt6['Age_new'] = bt6['Age_new'].astype(str)
bt6['Light_new'] = bt6['Light_new'].astype(str)

age_order = ['Young', 'Mid', 'Old']
age_pal = {'Young': '#55A868', 'Mid': '#4C72B0', 'Old': '#C44E52'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: By Age
ax = axes[0]
sns.boxplot(data=bt6, x='Age_new', y='probe_accuracy', order=age_order,
            palette=age_pal, width=0.5, fliersize=0, boxprops=dict(alpha=0.4), ax=ax)
sns.stripplot(data=bt6, x='Age_new', y='probe_accuracy', order=age_order,
              palette=age_pal, jitter=0.2, alpha=0.6, size=5, ax=ax)

# Kruskal-Wallis
groups = [bt6[bt6['Age_new'] == a]['probe_accuracy'].dropna() for a in age_order]
h, kw_p = stats.kruskal(*[g for g in groups if len(g) > 0])
ax.set_title(f'A. Probe accuracy by Age\nKruskal-Wallis p = {kw_p:.4f}', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Age Group')
ax.set_ylim(-0.02, 0.75)

# Panel B: Age x Light interaction
ax = axes[1]
sns.pointplot(data=bt6, x='Age_new', y='probe_accuracy', hue='Light_new',
              order=age_order, palette=PAL, errorbar='se', dodge=0.15,
              markers=['o', 's'], capsize=0.05, ax=ax)
ax.set_title('B. Age x Light interaction\n(OLS interaction p = 0.054)', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Age Group')
ax.set_ylim(0, 0.4)
ax.legend(title='Light Group')

fig.suptitle('Age is the Dominant Predictor of Spatial Learning',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_age_effect_barnes.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4: Effect Sizes (Cohen's d) Forest Plot
Shows that all ISF vs CTR effects are small (|d| < 0.3)

In [ ]:
eff = effects.copy()
# Shorten labels
eff['label'] = eff['Outcome'].str.replace('Circadian ', '').str.replace('Barnes T6 ', 'Barnes: ')

fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(eff))
colors = ['#C44E52' if abs(d) >= 0.2 else '#4C72B0' for d in eff['Cohen_d']]

ax.barh(y_pos, eff['Cohen_d'], color=colors, alpha=0.7, height=0.7, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.axvline(0.2, color='grey', ls=':', lw=1, alpha=0.5)
ax.axvline(-0.2, color='grey', ls=':', lw=1, alpha=0.5)
ax.axvline(0.5, color='red', ls='--', lw=1, alpha=0.5)
ax.axvline(-0.5, color='red', ls='--', lw=1, alpha=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(eff['label'], fontsize=10)
ax.set_xlabel("Cohen's d (ISF - CTR)", fontsize=12)
ax.set_title("Effect Sizes: All ISF vs CTR Comparisons\nDotted = small (0.2), Dashed = medium (0.5)",
             fontsize=14, fontweight='bold')

# Add text for d values
for i, (d, label) in enumerate(zip(eff['Cohen_d'], eff['label'])):
    ax.text(d + 0.02 * np.sign(d), i, f'd={d:.2f}', va='center', fontsize=9)

ax.set_xlim(-0.6, 0.6)
fig.tight_layout()
fig.savefig('figures/fig_effect_sizes_forest.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 5: Statistical Power
Shows that the study was underpowered to detect medium effects (d=0.5)

In [ ]:
eff = effects.copy()
eff['label'] = eff['Outcome'].str.replace('Circadian ', '').str.replace('Barnes T6 ', 'Barnes: ')

fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(eff))
bar_width = 0.35

bars1 = ax.barh(y_pos - bar_width/2, eff['Power_observed'], height=bar_width,
                color='#4C72B0', alpha=0.7, label='Power for observed d')
bars2 = ax.barh(y_pos + bar_width/2, eff['Power_d05'], height=bar_width,
                color='#DD8452', alpha=0.7, label='Power for d = 0.5')

ax.axvline(0.80, color='red', ls='--', lw=2, label='80% power threshold')

ax.set_yticks(y_pos)
ax.set_yticklabels(eff['label'], fontsize=10)
ax.set_xlabel('Statistical Power', fontsize=12)
ax.set_xlim(0, 1.0)
ax.set_title('Post-hoc Power Analysis\nAll outcomes underpowered to detect medium effects (d=0.5)',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)

fig.tight_layout()
fig.savefig('figures/fig_power_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 6: Learning Curve by Age Group
Shows that age modulates learning trajectory

In [ ]:
lc = barnes.dropna(subset=['probe_accuracy']).copy()
lc['Age_new'] = lc['Age_new'].astype(str)
lc['Light_new'] = lc['Light_new'].astype(str)

age_pal = {'Young': '#55A868', 'Mid': '#4C72B0', 'Old': '#C44E52'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Learning curve by Age
ax = axes[0]
sns.pointplot(data=lc, x='Trial', y='probe_accuracy', hue='Age_new',
              hue_order=['Young', 'Mid', 'Old'], palette=age_pal,
              errorbar='se', dodge=0.15, markers=['o', 's', 'D'],
              capsize=0.05, ax=ax)
ax.set_title('A. Learning curve by Age group', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Trial')
ax.set_ylim(0, 0.65)
ax.legend(title='Age Group')

# Panel B: Learning curve by Age x Light (faceted)
ax = axes[1]
# Combine Age + Light for hue
lc['Age_Light'] = lc['Age_new'] + ' ' + lc['Light_new']
combo_pal = {
    'Young CTR': '#55A868', 'Young ISF': '#88D8A8',
    'Mid CTR': '#4C72B0', 'Mid ISF': '#8CB4E0',
    'Old CTR': '#C44E52', 'Old ISF': '#E8A0A0',
}
sns.pointplot(data=lc, x='Trial', y='probe_accuracy', hue='Age_Light',
              hue_order=['Young CTR', 'Young ISF', 'Mid CTR', 'Mid ISF', 'Old CTR', 'Old ISF'],
              palette=combo_pal, errorbar='se', dodge=0.2, ax=ax,
              markers=['o', 'o', 's', 's', 'D', 'D'], capsize=0.03)
ax.set_title('B. Learning curve by Age x Light', fontsize=13)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Trial')
ax.set_ylim(0, 0.65)
ax.legend(title='Group', fontsize=8, title_fontsize=9, ncol=2)

fig.suptitle('Age Modulates Learning Trajectory; 40 Hz Light Does Not',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_learning_by_age.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary Slide: Key Takeaways

1. **Probe accuracy** (target zone entries) has real variance; goal-box accuracy is at floor
2. **No ISF effect** on Trial 6 accuracy (Mann-Whitney p=0.36, OLS p=0.73)
3. **No ISF effect** on learning rate (Trial x Light p=0.24)
4. **Age is the dominant factor**: Young > Mid > Old (Kruskal-Wallis p=0.0008)
5. **All effect sizes are small** (median |d| = 0.13)
6. **Study was underpowered** for medium effects (power ~0.65 for d=0.5)
7. These results are **consistent** with the original analysis — improved endpoint does not change conclusions